# 0. Data Generation

Before delving into any of the matters, its crucial to get or generate the needed data for this procedure. The two essential data forms for the RLHF process (excluding SFT) are, for Reward model training (1) and for reinforcement learning fine-tuning loop:

$$
\mathcal{D}_{\text{RM}} = \left\{(x^{(i)}, y^{(i)}_c, y^{(i)}_r)\right\}_{i=1}^N
$$

where $x^{(i)}$ is the prompt, $y^{(i)}_c$ is the chosen (preferred) response, and $y^{(i)}_r$ is the rejected (non-preferred) response.

$$
\mathcal{D}_{\text{RL}} = \left\{(x^{(j)}, y^{(j)}, R^{(j)})\right\}_{j=1}^M

$$
where $x^{(j)}$ is the prompt, $y^{(j)}$ is the generated response (trajectory), and $R^{(j)}$ is the scalar reward, $R^{(j)} = \mathbf{r}_\psi(x^{(j)}, y^{(j)})$.


Obviously the first model needs to be split into train/test/eval data to avoid overfitting human preference or falling into **reward hacking**. The second dataset is not a static dataset (Only the user prompts are static), given that the rewards and trajectory are computed online during the RL process.

> To achieve this we need to gather a set of base user prompts (Used for both tasks) and manually curate a set of 1000-5000 Approved/Rejected pairs. 


In [3]:
import os
import datasets
from loguru import logger

In [14]:
def create_sample_dataset(
    full_dataset_name: str,
    new_name: str = "",
    sample_count: int = 20000,
    username: str = "eZWALT",
    cache_dir: str = "./dataset",
    split_percentage: float = 0.8,
    columns_to_keep: list[str] | None = None,
):
    os.makedirs(cache_dir, exist_ok=True)

    # Derive sample dataset name
    dataset_name = full_dataset_name.split("/")[-1]
    dataset_name_sample = f"{dataset_name}-sample-{sample_count}" if new_name == "" else new_name
    repo_id = f"{username}/{dataset_name_sample}"

    # --- Load both splits from Hugging Face ---
    try:
        train_ds = datasets.load_dataset(full_dataset_name, cache_dir=cache_dir, split="train")
        test_ds = datasets.load_dataset(full_dataset_name, cache_dir=cache_dir, split="test")
    except Exception as e:
        raise ValueError(f"Could not load train/test splits for {full_dataset_name}: {e}")

    # --- Keep only specific columns if provided ---
    if columns_to_keep:
        for split_name, split_ds in {"train": train_ds, "test": test_ds}.items():
            all_cols = split_ds.column_names
            cols_to_drop = list(set(all_cols) - set(columns_to_keep))
            if cols_to_drop:
                split_ds = split_ds.remove_columns(cols_to_drop)
                logger.info(f"{split_name}: Kept columns {split_ds.column_names}")
            if split_name == "train":
                train_ds = split_ds
            else:
                test_ds = split_ds

    # --- Sample each split separately ---
    train_n = int(sample_count * split_percentage)
    test_n = sample_count - train_n

    if train_n > len(train_ds):
        logger.warning(f"Requested {train_n} train samples but dataset has only {len(train_ds)}.")
        train_n = len(train_ds)
    if test_n > len(test_ds):
        logger.warning(f"Requested {test_n} test samples but dataset has only {len(test_ds)}.")
        test_n = len(test_ds)

    train_sample = train_ds.shuffle(seed=42).select(range(train_n))
    test_sample = test_ds.shuffle(seed=42).select(range(test_n))

    # --- Push both splits to the Hub ---
    try:
        train_sample.push_to_hub(repo_id, split="train")
        logger.info("✅ Train split pushed to the hub successfully.")
    except Exception as e:
        logger.warning(f"❌ Failed to push train split: {e}")

    try:
        test_sample.push_to_hub(repo_id, split="test")
        logger.info("✅ Test split pushed to the hub successfully.")
    except Exception as e:
        logger.warning(f"❌ Failed to push test split: {e}")

    return {"train": train_sample, "test": test_sample}

In [15]:
create_sample_dataset(
    "yitingxie/rlhf-reward-datasets",
    "rlhf_user_prompts",
    sample_count=25000,
    split_percentage=0.8,
    columns_to_keep=["prompt"]
)

2025-10-15 11:52:15.870 | INFO     | __main__:create_sample_dataset:31 - train: Kept columns ['prompt']
2025-10-15 11:52:15.875 | INFO     | __main__:create_sample_dataset:31 - test: Kept columns ['prompt']
Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.02s/ shards]
2025-10-15 11:52:20.766 | INFO     | __main__:create_sample_dataset:54 - ✅ Train split pushed to the hub successfully.
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.74s/ shards]
2025-10-15 11:52:24.704 | INFO     | __main__:create_sample_dataset:60 - ✅ Test split pushed to the hub successfully.


{'train': Dataset({
     features: ['prompt'],
     num_rows: 20000
 }),
 'test': Dataset({
     features: ['prompt'],
     num_rows: 5000
 })}

## Dataset usage

Now that we sucessfully acomplished to get a database of prompts and we pushed it to hugging face, this is going to enable the next steps of this process. In order to reproduce this a hugging face account is needed and huggingface-cli configuration with its respective access token. 